[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/codex/laya-fp16-fast-inference/notebooks/Laya_FP16_All28_BruteForce_V3_Colab.ipynb)

# Laya FP16 V3 — all-28-layer brute-force search

**Model:** `vtava/Laya-1.2-speedup-standalone-fp16` · **GPU runtime required**

Your V2 T4 run reduced the complete API from **34.43 to 12.55 ms (2.74×)**.
It beat the already-compiled original by **16%** on those three synthetic API cases.
Most tensor-forward speedup came from compilation. All recorded parity checks passed;
these were not gold-label accuracy tests.

V3 now changes the encoder computation: **140 single-layer trials**, four all-layer
trials, bounded pairs and cumulative combinations. It tries attention removal, FFN
removal, full-block bypass and activation-ranked FFN channel pruning (75%/50% targets).
All 28 layers are searched. This is not an exhaustive search over all 6^28 architectures.

**Run all** with a GPU. Hundreds of evaluations and compilation can take substantial
time. No perfect result or further speedup is guaranteed. The unchanged model can win.
No model is uploaded automatically.

In [ ]:
#@title 1. Install pinned libraries and fetch the exact implementation
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
SOURCE_REVISION = 'ffe5f337736e83c5741e55bbf3ff849f1e6cdf7d'
MODEL_REVISION = '2891f7312d8e7c65b43a6e04c29cc226b06b4fac'
DATASET_REVISION = '7189f46b02248bd2569df7325c9df8e3b24e90a7'
REPO = Path(tempfile.mkdtemp(prefix='laya-v3-')) / 'TinyCeNN-LM'
REPO.mkdir()
subprocess.run(['git','init','-q',str(REPO)],check=True)
subprocess.run(['git','remote','add','origin','https://github.com/vtavakkoli/TinyCeNN-LM.git'],cwd=REPO,check=True)
subprocess.run(['git','fetch','--depth','1','origin',SOURCE_REVISION],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO,check=True)
print('Pinned source:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

## Data and quality rules

The labelled `LocalLLaMA/typed-decisions` benchmark is itself synthetic, but provides
actual gold labels and all three question types. Calibration, search and validation
use disjoint cases from `train`. The official `test` split is loaded only after the
candidate is frozen. Earlier checkpoint training overlap is unknown.

Default gates: **≥98% agreement**, **≤0.5 percentage points overall accuracy loss**,
**≤2 points per workflow/type**, and mean JS ≤0.005. A changed model must be at least
3% faster than the **optimized compiled baseline**, with the same workload and compiler
settings. The final report also shows a paired case-bootstrap accuracy interval.

The search physically shrinks FFN matrices, retaining paired GLU channels. Attention
uses native PyTorch SDPA. Compiler-generated kernels may use Triton internally.
No recovery training is performed.

In [ ]:
#@title 2. Search budgets and resumable output
SAVE_TO_DRIVE = False #@param {type:"boolean"}
RUN_NAME = 'laya_fp16_all28_v3' #@param {type:"string"}
CALIBRATION_CASES = 24 #@param {type:"integer"}
SEARCH_CASES = 32 #@param {type:"integer"}
VALIDATION_CASES = 160 #@param {type:"integer"}
PAIR_TOP = 12 #@param {type:"integer"}
COMBINE_TOP = 24 #@param {type:"integer"}
BEAM_WIDTH = 2 #@param {type:"integer"}
COMBINATION_DEPTH = 6 #@param {type:"integer"}
VALIDATION_TOP = 12 #@param {type:"integer"}
COMPILED_FINALISTS = 3 #@param {type:"integer"}
MIN_AGREEMENT = 0.98 #@param {type:"number"}
MAX_ACCURACY_DROP = 0.005 #@param {type:"number"}
MAX_GROUP_DROP = 0.02 #@param {type:"number"}
MIN_SPEEDUP = 1.03 #@param {type:"number"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/LayaExperiments') / RUN_NAME
else:
    OUT = Path('/content') / RUN_NAME
OUT.mkdir(parents=True,exist_ok=True)
print('Output:',OUT)
print('Same settings + same folder resumes completed trials. Changed settings require a new RUN_NAME.')

In [ ]:
#@title 3. Run all-layer trials, combinations, validation and one final test audit
cmd = [sys.executable,'-u',str(REPO/'scripts/search_laya_layers_v3.py'),
       '--output',str(OUT),'--dataset-revision',DATASET_REVISION,
       '--calibration',str(CALIBRATION_CASES),'--screen',str(SEARCH_CASES),
       '--validation',str(VALIDATION_CASES),'--pair-top',str(PAIR_TOP),
       '--combine-top',str(COMBINE_TOP),'--beam',str(BEAM_WIDTH),
       '--depth',str(COMBINATION_DEPTH),'--validation-top',str(VALIDATION_TOP),
       '--finalists',str(COMPILED_FINALISTS),'--min-agreement',str(MIN_AGREEMENT),
       '--max-accuracy-drop',str(MAX_ACCURACY_DROP),'--max-group-drop',str(MAX_GROUP_DROP),
       '--min-speedup',str(MIN_SPEEDUP)]
log_path=OUT/'run.log'
process=None
try:
    with log_path.open('a',buffering=1) as log:
        process=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,bufsize=1,env=os.environ.copy())
        for line in process.stdout:
            print(line,end=''); log.write(line)
        status=process.wait()
    if status:
        raise RuntimeError(f'Search exited with {status}. Inspect {log_path}; completed trials are saved.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
    print('Stopped. Completed trials remain saved; rerun this cell to resume.')
    raise

In [ ]:
#@title 4. Inspect measured results (also works with a partial run)
import pandas as pd
from IPython.display import display
trial_path=OUT/'trials.json'
if trial_path.exists():
    trials=json.loads(trial_path.read_text())
    table=[]
    for r in trials.values():
        m=r.get('metrics',{})
        table.append(dict(phase=r['phase'],changed_layers=len(r['plan']),passed=r['quality_pass'],
                          agreement=m.get('agreement'),accuracy=m.get('accuracy'),
                          accuracy_delta=m.get('accuracy_delta'),JS=m.get('mean_js'),
                          parameters=r.get('parameters'),status=r['status']))
    display(pd.DataFrame(table).sort_values(['passed','parameters'],ascending=[False,True]).head(30))
report_path=OUT/'final_report.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print('OUTCOME:',report['outcome'])
    print('Changed layers:',report['changed_layers'],'| Recommended:',report['recommended'])
    audit=report['final_audit']
    display(pd.DataFrame([{k:audit[k] for k in ['cases','decisions','accuracy','reference_accuracy',
                                              'accuracy_delta','agreement','mean_js']}]))
    print('Paired case accuracy-delta 95% interval:',audit['paired_case_accuracy_delta_95pct_interval'])
    api=report['end_to_end_api']
    display(pd.DataFrame(api['results']).T)
    print('API timing:',api['unit'])
    if api['errors']: print('API failures:',api['errors'])
    print('Recipe:',OUT/'selected_recipe.json')
else:
    print('No final report yet. Partial trial results can still be downloaded below.')

In [ ]:
#@title 5. Download results and reconstructable architecture recipe
from google.colab import files
assert OUT.exists(), 'Run setup first.'
archive=shutil.make_archive('/content/laya_fp16_v3_results','zip',root_dir=OUT)
print('Includes reports, trial history, log, exact channel indices and loader code.')
print('The recipe requires the pinned original checkpoint; it is not a standalone weights export.')
files.download(archive)

## Load the selected configuration later

Extract the ZIP, install `transformers==4.57.6`, `huggingface_hub<1`, `safetensors`
and a compatible CUDA PyTorch. Run from that directory:

```python
from layer_search import load_selected
model, tokenizer, runtime = load_selected('selected_recipe.json', device='cuda')
```

The loader recreates the recorded channel indices and layer changes from the pinned
original weights. Check `recommended` in the recipe first. Failed final audits remain
experimental; `no_qualifying_layer_change` returns the unchanged architecture.

Upload `laya_fp16_v3_results.zip` for review of the next measured result.